In [1]:
# 1. Imports
import sys
sys.path.append("..")

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from src.preprocessing import preprocess

# 2. Load raw data
raw_train = pd.read_csv("../data/raw/train.csv")
raw_test = pd.read_csv("../data/raw/test.csv")

# 3. Compute the three "learned from train only" lookup tables, BEFORE any preprocessing call
temp_title = raw_train["Name"].str.extract(r",\s*([^.]+)\.")[0]
temp_title = temp_title.where(temp_title.isin(["Mr", "Miss", "Mrs", "Master"]), "Rare")
age_medians_by_title = raw_train["Age"].groupby(temp_title).median()

fare_medians_by_pclass = raw_train.groupby("Pclass")["Fare"].median()

combined_ticket_counts = pd.concat([raw_train["Ticket"], raw_test["Ticket"]]).value_counts()

# 4. NOW preprocess both train and test, using those same fixed lookup tables
train_processed = preprocess(raw_train, age_medians_by_title, combined_ticket_counts, fare_medians_by_pclass)
test_processed = preprocess(raw_test, age_medians_by_title, combined_ticket_counts, fare_medians_by_pclass)

# 5. Build X, y from the processed train data
feature_cols = [
    "IsFemale", "Pclass", "Age", "Fare", "FamilySize", "HasCabin",
    "Title_Master", "Title_Miss", "Title_Mr", "Title_Mrs", "Title_Rare",
    "Embarked_C", "Embarked_Q", "Embarked_S", "TicketGroupSize"
]
X = train_processed[feature_cols]
y = train_processed["Survived"]
X_test_final = test_processed[feature_cols]

# 6. Sanity check before training anything
print(X.isna().sum().sum(), "missing values in X")
print(X_test_final.isna().sum().sum(), "missing values in X_test_final")

0 missing values in X
0 missing values in X_test_final


In [2]:
final_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
final_model.fit(X, y)

test_predictions = final_model.predict(X_test_final)

In [3]:
submission = pd.DataFrame({
    "PassengerId": raw_test["PassengerId"],
    "Survived": test_predictions.astype(int)
})
submission.to_csv("../data/processed/submission.csv", index=False)

In [4]:
submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [5]:
submission.shape
submission["Survived"].value_counts()

Survived
0    271
1    147
Name: count, dtype: int64

In [6]:
test_processed[["Pclass", "IsFemale", "Title_Mr"]].describe(include="all")

,Pclass,IsFemale,Title_Mr
count,418.000000,418,418
unique,NaN,2,2
top,NaN,False,True
freq,NaN,266,240
mean,2.265550,NaN,NaN
std,0.841838,NaN,NaN
min,1.000000,NaN,NaN
25%,1.000000,NaN,NaN
50%,3.000000,NaN,NaN
75%,3.000000,NaN,NaN


In [7]:
from sklearn.linear_model import LogisticRegression

final_lr_model = LogisticRegression(max_iter=1000)
final_lr_model.fit(X, y)

lr_predictions = final_lr_model.predict(X_test_final)

lr_submission = pd.DataFrame({
    "PassengerId": raw_test["PassengerId"],
    "Survived": lr_predictions.astype(int)
})
lr_submission.to_csv("../data/processed/submission_logreg.csv", index=False)

In [8]:
train_processed["Fare"].corr(train_processed["TicketGroupSize"])

np.float64(0.43654407180369786)

In [9]:
train_processed[feature_cols + ["Survived"]].corr()

,IsFemale,Pclass,Age,Fare,FamilySize,HasCabin,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Embarked_C,Embarked_Q,Embarked_S,TicketGroupSize,Survived
IsFemale,1.000000,-0.131900,-0.104717,0.182333,0.200988,0.140391,-0.159934,0.686808,-0.867334,0.547600,-0.034471,0.082853,0.074115,-0.119224,0.176548,0.543351
Pclass,-0.131900,1.000000,-0.353813,-0.549500,0.065997,-0.725541,0.082081,-0.000576,0.142698,-0.149209,-0.206333,-0.243292,0.221009,0.074053,-0.039893,-0.338481
Age,-0.104717,-0.353813,1.000000,0.097939,-0.275474,0.243897,-0.407615,-0.296883,0.215990,0.194681,0.174374,0.040748,-0.062791,0.003817,-0.239378,-0.078698
Fare,0.182333,-0.549500,0.097939,1.000000,0.217138,0.482075,0.010908,0.118271,-0.183766,0.105203,0.024585,0.269335,-0.117216,-0.162184,0.436544,0.257307
FamilySize,0.200988,0.065997,-0.275474,0.217138,1.000000,-0.009175,0.372472,0.112838,-0.338014,0.156168,-0.058565,-0.046215,-0.058592,0.077359,0.816020,0.016639
HasCabin,0.140391,-0.725541,0.243897,0.482075,-0.009175,1.000000,-0.027841,0.035314,-0.137319,0.118300,0.106246,0.208528,-0.129572,-0.101139,0.055447,0.316912
Title_Master,-0.159934,0.082081,-0.407615,0.010908,0.372472,-0.027841,1.000000,-0.109844,-0.254903,-0.087580,-0.038326,-0.035225,0.010478,0.024264,0.323431,0.085221
Title_Miss,0.686808,-0.000576,-0.296883,0.118271,0.112838,0.035314,-0.109844,1.000000,-0.595692,-0.204670,-0.089565,0.026215,0.171117,-0.130650,0.127032,0.327093
Title_Mr,-0.867334,0.142698,0.215990,-0.183766,-0.338014,-0.137319,-0.254903,-0.595692,1.000000,-0.474952,-0.207843,-0.072567,-0.078338,0.112870,-0.288334,-0.549199
Title_Mrs,0.547600,-0.149209,0.194681,0.105203,0.156168,0.118300,-0.087580,-0.204670,-0.474952,1.000000,-0.071411,0.061395,-0.089739,0.002689,0.102312,0.339040


In [10]:
correlations = train_processed[feature_cols + ["Survived"]].corr()["Survived"].sort_values(ascending=False)
correlations

Survived           1.000000
IsFemale           0.543351
Title_Mrs          0.339040
Title_Miss         0.327093
HasCabin           0.316912
Fare               0.257307
Embarked_C         0.168240
Title_Master       0.085221
TicketGroupSize    0.064962
Title_Rare         0.022030
FamilySize         0.016639
Embarked_Q         0.003650
Age               -0.078698
Embarked_S        -0.149683
Pclass            -0.338481
Title_Mr          -0.549199
Name: Survived, dtype: float64

In [11]:
train_processed[["Fare", "TicketGroupSize", "FamilySize", "Survived"]].corr()

,Fare,TicketGroupSize,FamilySize,Survived
Fare,1.000000,0.436544,0.217138,0.257307
TicketGroupSize,0.436544,1.000000,0.816020,0.064962
FamilySize,0.217138,0.816020,1.000000,0.016639
Survived,0.257307,0.064962,0.016639,1.000000


In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv_shuffled = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42), X, y, cv=cv_shuffled)
print(f"mean={scores.mean()*100:.2f}%, std={scores.std()*100:.2f}")

mean=84.40%, std=0.92


In [13]:
final_check_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
final_check_model.fit(X, y)

pd.Series(final_check_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

Title_Mr           0.179290
IsFemale           0.141103
Fare               0.140185
Age                0.113870
Pclass             0.084703
TicketGroupSize    0.076318
FamilySize         0.059105
Title_Mrs          0.054450
Title_Miss         0.049428
HasCabin           0.044298
Title_Master       0.013454
Embarked_C         0.013231
Embarked_S         0.013229
Title_Rare         0.008718
Embarked_Q         0.008617
dtype: float64